## Importando dados e criando tabelas

Observando os catálogos

In [0]:
%sql
SHOW CATALOGS

catalog
dbacademy
samples
system
workspace


In [0]:
%sql
SHOW VOLUMES;


database,volume_name
default,dados


Criando volume na workspace

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.dados
--/Volumes/workspace/defaut/dados

Importando dados e definindo schema

In [0]:
from pyspark.sql.functions import *
arqschema = "id INT, nome STRING, status STRING, cidade STRING, vendas INT, data STRING"

# Caminho do arquvivo no volume
despachantes = spark.read.csv("/Volumes/workspace/default/dados/despachantes.csv", header = False, schema = arqschema)
despachantes.show()

+---+-------------------+------+-------------+------+----------+
| id|               nome|status|       cidade|vendas|      data|
+---+-------------------+------+-------------+------+----------+
|  1|   Carminda Pestana| Ativo|  Santa Maria|    23|2020-08-11|
|  2|    Deolinda Vilela| Ativo|Novo Hamburgo|    34|2020-03-05|
|  3|   Emídio Dornelles| Ativo| Porto Alegre|    34|2020-02-05|
|  4|Felisbela Dornelles| Ativo| Porto Alegre|    36|2020-02-05|
|  5|     Graça Ornellas| Ativo| Porto Alegre|    12|2020-02-05|
|  6|   Matilde Rebouças| Ativo| Porto Alegre|    22|2019-01-05|
|  7|    Noêmia   Orriça| Ativo|  Santa Maria|    45|2019-10-05|
|  8|      Roque Vásquez| Ativo| Porto Alegre|    65|2020-03-05|
|  9|      Uriel Queiroz| Ativo| Porto Alegre|    54|2018-05-05|
| 10|   Viviana Sequeira| Ativo| Porto Alegre|     0|2020-09-05|
+---+-------------------+------+-------------+------+----------+



Outro exemplo, inferindo schema, usando load e informando tipo

In [0]:
desp_autoschema = spark.read.load("/Volumes/workspace/default/dados/despachantes.csv",
                                  format = "csv", sep = ",", inferschema = True, header = False)
desp_autoschema.show()

+---+-------------------+-----+-------------+---+----------+
|_c0|                _c1|  _c2|          _c3|_c4|       _c5|
+---+-------------------+-----+-------------+---+----------+
|  1|   Carminda Pestana|Ativo|  Santa Maria| 23|2020-08-11|
|  2|    Deolinda Vilela|Ativo|Novo Hamburgo| 34|2020-03-05|
|  3|   Emídio Dornelles|Ativo| Porto Alegre| 34|2020-02-05|
|  4|Felisbela Dornelles|Ativo| Porto Alegre| 36|2020-02-05|
|  5|     Graça Ornellas|Ativo| Porto Alegre| 12|2020-02-05|
|  6|   Matilde Rebouças|Ativo| Porto Alegre| 22|2019-01-05|
|  7|    Noêmia   Orriça|Ativo|  Santa Maria| 45|2019-10-05|
|  8|      Roque Vásquez|Ativo| Porto Alegre| 65|2020-03-05|
|  9|      Uriel Queiroz|Ativo| Porto Alegre| 54|2018-05-05|
| 10|   Viviana Sequeira|Ativo| Porto Alegre|  0|2020-09-05|
+---+-------------------+-----+-------------+---+----------+



Comparando os schemas

In [0]:
desp_autoschema.printSchema()
despachantes.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: integer (nullable = true)
 |-- _c5: date (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- status: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- vendas: integer (nullable = true)
 |-- data: string (nullable = true)



Usando pyspark e fazendo algumas operações sobre o dataframe

In [0]:
from pyspark.sql import functions as f

# condição lógica com where
despachantes.select("id", "nome", "vendas").where(f.col("vendas") > 20).display()

# & para and | para or, e ~ para not
despachantes.select("id", "nome", "vendas").where((f.col("vendas") > 20) & (f.col("vendas") < 40)).display()

id,nome,vendas
1,Carminda Pestana,23
2,Deolinda Vilela,34
3,Emídio Dornelles,34
4,Felisbela Dornelles,36
6,Matilde Rebouças,22
7,Noêmia Orriça,45
8,Roque Vásquez,65
9,Uriel Queiroz,54


id,nome,vendas
1,Carminda Pestana,23
2,Deolinda Vilela,34
3,Emídio Dornelles,34
4,Felisbela Dornelles,36
6,Matilde Rebouças,22


Renomeando coluna

In [0]:
novodf = despachantes.withColumnRenamed("nome", "nomes").withColumnRenamed("vendas", "total_vendas")
novodf.columns

['id', 'nomes', 'status', 'cidade', 'total_vendas', 'data']

Tratando a data

In [0]:
from pyspark.sql.functions import *
# A coluna data está como string, vamos transformar em data
despachantes2 = despachantes.withColumn("data2", to_timestamp(f.col("data"), "yyyy-MM-dd"))
despachantes2.printSchema()

root
 |-- id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- status: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- vendas: integer (nullable = true)
 |-- data: string (nullable = true)
 |-- data2: timestamp (nullable = true)



In [0]:
# Operaçõe sobre data
despachantes2.select(year("data")).show()
despachantes2.select(year("data")).distinct().show()
despachantes2.select("nome", year("data")).orderBy("nome").show()
despachantes2.select("data").groupBy(year("data")).count().show()
despachantes2.select(f.sum("vendas")).show()

+----------+
|year(data)|
+----------+
|      2020|
|      2020|
|      2020|
|      2020|
|      2020|
|      2019|
|      2019|
|      2020|
|      2018|
|      2020|
+----------+

+----------+
|year(data)|
+----------+
|      2019|
|      2020|
|      2018|
+----------+

+-------------------+----------+
|               nome|year(data)|
+-------------------+----------+
|   Carminda Pestana|      2020|
|    Deolinda Vilela|      2020|
|   Emídio Dornelles|      2020|
|Felisbela Dornelles|      2020|
|     Graça Ornellas|      2020|
|   Matilde Rebouças|      2019|
|    Noêmia   Orriça|      2019|
|      Roque Vásquez|      2020|
|      Uriel Queiroz|      2018|
|   Viviana Sequeira|      2020|
+-------------------+----------+

+----------+-----+
|year(data)|count|
+----------+-----+
|      2019|    2|
|      2020|    7|
|      2018|    1|
+----------+-----+

+-----------+
|sum(vendas)|
+-----------+
|        325|
+-----------+



Salvando tipos, são diretórios

In [0]:
despachantes.write.mode("overwrite").format("parquet").save("/Volumes/workspace/default/dados/dfimportparquet")
despachantes.write.mode("overwrite").format("csv").save("/Volumes/workspace/default/dados/dfimportcsv")
despachantes.write.mode("overwrite").format("json").save("/Volumes/workspace/default/dados/dfimportpjson")
despachantes.write.mode("overwrite").format("orc").save("/Volumes/workspace/default/dados/dfimportorc")

Lendo dados

In [0]:
par = spark.read.format("parquet").load("/Volumes/workspace/default/dados/dfimportparquet/")
par.show()
par.printSchema()

+---+-------------------+------+-------------+------+----------+
| id|               nome|status|       cidade|vendas|      data|
+---+-------------------+------+-------------+------+----------+
|  1|   Carminda Pestana| Ativo|  Santa Maria|    23|2020-08-11|
|  2|    Deolinda Vilela| Ativo|Novo Hamburgo|    34|2020-03-05|
|  3|   Emídio Dornelles| Ativo| Porto Alegre|    34|2020-02-05|
|  4|Felisbela Dornelles| Ativo| Porto Alegre|    36|2020-02-05|
|  5|     Graça Ornellas| Ativo| Porto Alegre|    12|2020-02-05|
|  6|   Matilde Rebouças| Ativo| Porto Alegre|    22|2019-01-05|
|  7|    Noêmia   Orriça| Ativo|  Santa Maria|    45|2019-10-05|
|  8|      Roque Vásquez| Ativo| Porto Alegre|    65|2020-03-05|
|  9|      Uriel Queiroz| Ativo| Porto Alegre|    54|2018-05-05|
| 10|   Viviana Sequeira| Ativo| Porto Alegre|     0|2020-09-05|
+---+-------------------+------+-------------+------+----------+

root
 |-- id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- status: 

Outras formas de ler os dados

In [0]:
# FORMATO DE LISTA
despachantes.take(2)

[Row(id=1, nome='Carminda Pestana', status='Ativo', cidade='Santa Maria', vendas=23, data='2020-08-11'),
 Row(id=2, nome='Deolinda Vilela', status='Ativo', cidade='Novo Hamburgo', vendas=34, data='2020-03-05')]

In [0]:
# RETORNA TODOS OS DADOS COMO UMA LISTA
despachantes.collect()

[Row(id=1, nome='Carminda Pestana', status='Ativo', cidade='Santa Maria', vendas=23, data='2020-08-11'),
 Row(id=2, nome='Deolinda Vilela', status='Ativo', cidade='Novo Hamburgo', vendas=34, data='2020-03-05'),
 Row(id=3, nome='Emídio Dornelles', status='Ativo', cidade='Porto Alegre', vendas=34, data='2020-02-05'),
 Row(id=4, nome='Felisbela Dornelles', status='Ativo', cidade='Porto Alegre', vendas=36, data='2020-02-05'),
 Row(id=5, nome='Graça Ornellas', status='Ativo', cidade='Porto Alegre', vendas=12, data='2020-02-05'),
 Row(id=6, nome='Matilde Rebouças', status='Ativo', cidade='Porto Alegre', vendas=22, data='2019-01-05'),
 Row(id=7, nome='Noêmia   Orriça', status='Ativo', cidade='Santa Maria', vendas=45, data='2019-10-05'),
 Row(id=8, nome='Roque Vásquez', status='Ativo', cidade='Porto Alegre', vendas=65, data='2020-03-05'),
 Row(id=9, nome='Uriel Queiroz', status='Ativo', cidade='Porto Alegre', vendas=54, data='2018-05-05'),
 Row(id=10, nome='Viviana Sequeira', status='Ativo', c

In [0]:
# CONTA O NÚMERO DE REGISTROS
despachantes.count()

10

In [0]:
# TRANSFORMAÇÕES, PADRÃO DECRESCENTE
despachantes.orderBy("vendas").show()

+---+-------------------+------+-------------+------+----------+
| id|               nome|status|       cidade|vendas|      data|
+---+-------------------+------+-------------+------+----------+
| 10|   Viviana Sequeira| Ativo| Porto Alegre|     0|2020-09-05|
|  5|     Graça Ornellas| Ativo| Porto Alegre|    12|2020-02-05|
|  6|   Matilde Rebouças| Ativo| Porto Alegre|    22|2019-01-05|
|  1|   Carminda Pestana| Ativo|  Santa Maria|    23|2020-08-11|
|  3|   Emídio Dornelles| Ativo| Porto Alegre|    34|2020-02-05|
|  2|    Deolinda Vilela| Ativo|Novo Hamburgo|    34|2020-03-05|
|  4|Felisbela Dornelles| Ativo| Porto Alegre|    36|2020-02-05|
|  7|    Noêmia   Orriça| Ativo|  Santa Maria|    45|2019-10-05|
|  9|      Uriel Queiroz| Ativo| Porto Alegre|    54|2018-05-05|
|  8|      Roque Vásquez| Ativo| Porto Alegre|    65|2020-03-05|
+---+-------------------+------+-------------+------+----------+



In [0]:
# PADÃO DECRESCENTE
despachantes.orderBy(f.col("vendas").desc()).show()

+---+-------------------+------+-------------+------+----------+
| id|               nome|status|       cidade|vendas|      data|
+---+-------------------+------+-------------+------+----------+
|  8|      Roque Vásquez| Ativo| Porto Alegre|    65|2020-03-05|
|  9|      Uriel Queiroz| Ativo| Porto Alegre|    54|2018-05-05|
|  7|    Noêmia   Orriça| Ativo|  Santa Maria|    45|2019-10-05|
|  4|Felisbela Dornelles| Ativo| Porto Alegre|    36|2020-02-05|
|  2|    Deolinda Vilela| Ativo|Novo Hamburgo|    34|2020-03-05|
|  3|   Emídio Dornelles| Ativo| Porto Alegre|    34|2020-02-05|
|  1|   Carminda Pestana| Ativo|  Santa Maria|    23|2020-08-11|
|  6|   Matilde Rebouças| Ativo| Porto Alegre|    22|2019-01-05|
|  5|     Graça Ornellas| Ativo| Porto Alegre|    12|2020-02-05|
| 10|   Viviana Sequeira| Ativo| Porto Alegre|     0|2020-09-05|
+---+-------------------+------+-------------+------+----------+



In [0]:
# SE QUISERMOS DUAS COLUNAS EM FORMATO DECRESCENTE
despachantes.orderBy(f.col("cidade").desc(), (f.col("vendas").desc())).show()

+---+-------------------+------+-------------+------+----------+
| id|               nome|status|       cidade|vendas|      data|
+---+-------------------+------+-------------+------+----------+
|  7|    Noêmia   Orriça| Ativo|  Santa Maria|    45|2019-10-05|
|  1|   Carminda Pestana| Ativo|  Santa Maria|    23|2020-08-11|
|  8|      Roque Vásquez| Ativo| Porto Alegre|    65|2020-03-05|
|  9|      Uriel Queiroz| Ativo| Porto Alegre|    54|2018-05-05|
|  4|Felisbela Dornelles| Ativo| Porto Alegre|    36|2020-02-05|
|  3|   Emídio Dornelles| Ativo| Porto Alegre|    34|2020-02-05|
|  6|   Matilde Rebouças| Ativo| Porto Alegre|    22|2019-01-05|
|  5|     Graça Ornellas| Ativo| Porto Alegre|    12|2020-02-05|
| 10|   Viviana Sequeira| Ativo| Porto Alegre|     0|2020-09-05|
|  2|    Deolinda Vilela| Ativo|Novo Hamburgo|    34|2020-03-05|
+---+-------------------+------+-------------+------+----------+



In [0]:
# AGRUPAR DADOS POR VENDAS DECRESCENTE
# VER VENDAS POR CIDADE
despachantes.groupBy("cidade").agg(f.sum("vendas").alias("total_vendas_cidades")).show()

+-------------+--------------------+
|       cidade|total_vendas_cidades|
+-------------+--------------------+
|Novo Hamburgo|                  34|
| Porto Alegre|                 223|
|  Santa Maria|                  68|
+-------------+--------------------+



In [0]:
# ORDENAR POR VENDAS DECRESCENTE
despachantes.groupBy("cidade").agg(f.sum("vendas").alias("total_vendas_cidade")).orderBy(f.col("total_vendas_cidade").desc()).show()

+-------------+-------------------+
|       cidade|total_vendas_cidade|
+-------------+-------------------+
| Porto Alegre|                223|
|  Santa Maria|                 68|
|Novo Hamburgo|                 34|
+-------------+-------------------+



In [0]:
# FILTRAR (FILTER)
despachantes.filter(f.col("nome") == "Deolinda%").show()
# despachantes.filter(f.col("nome").like("Deolinda%")).show()

+---+---------------+------+-------------+------+----------+
| id|           nome|status|       cidade|vendas|      data|
+---+---------------+------+-------------+------+----------+
|  2|Deolinda Vilela| Ativo|Novo Hamburgo|    34|2020-03-05|
+---+---------------+------+-------------+------+----------+



Criando tabelas gerenciadas e persistidas no catálogo

In [0]:
despachantes.write.saveAsTable("despachantes")